**create a table and ingest dimensional data**

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

**Brands**

**define schema for brand table**

In [0]:
catalog_name='ecommerce'

# Define schema for data file

brand_schema= StructType([
    StructField('brand_code',StringType(),False),
    StructField('brand_name',StringType(),True),
    StructField('category_code',StringType(),True),
])

In [0]:
rawdata_path="/Volumes/ecommerce/source_data/raw/brands/*.csv"

df=spark.read.option('header',"true").option('delimiter',",").schema(brand_schema).csv(rawdata_path)

# add metadata columns
df=df.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())

display(df.limit(5))

**loading data from csv to spark dataframe**
#### now from spark df to delta table 

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .saveAsTable(f"{catalog_name}.bronze.brz_brands")

**Repeat the same writing to delta table for all dimenison data**
### Category

In [0]:
#schema 

category_schema=StructType([
    StructField("category_code",StringType(),True),
    StructField("category_name",StringType(),True),
])

raw_path="/Volumes/ecommerce/source_data/raw/category/*.csv"

#spark df

dfc=spark.read.option("header","true").option("delimiter",",").schema(category_schema).csv(raw_path)

#adding metadata

dfc=dfc.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())

display(dfc.limit(5))

#delta format

dfc.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .saveAsTable(f"{catalog_name}.bronze.brz_category")

**Products**

In [0]:
#schema

products_schema=StructType([
    StructField("product_id",StringType(),True),
    StructField("sku",StringType(),True),
    StructField("category_code",StringType(),True),
    StructField("brand_code",StringType(),True),
    StructField("color",StringType(),True),
    StructField("size",StringType(),True),
    StructField("material",StringType(),True),
    StructField("weight_grams",StringType(),True),
    StructField("length_cm",StringType(),True),
    StructField("width_cm",FloatType(),True),
    StructField("height_cm",FloatType(),True),
    StructField("rating_count", FloatType(),True),
])

raw_pr_path="/Volumes/ecommerce/source_data/raw/products/*.csv"

dfp=spark.read.option("header","true").option("delimiter",",").schema(products_schema).csv(raw_pr_path)

#coverting length from string to float and changing 22,2(it is in european format) to 22.2

dfp=dfp.withColumn("length_cm",F.regexp_replace(F.col("length_cm"),",",".").cast(FloatType())) \
    .withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())

display(dfp.limit(5))

dfp.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .saveAsTable(f"{catalog_name}.bronze.brz_products")


**Customers**

In [0]:
#schema

customers_schema=StructType([
    StructField("customer_id",StringType(),True),
    StructField("phone",FloatType(),True),
    StructField("country_code",StringType(),True),
    StructField("country",StringType(),True),
    StructField("state",StringType(),True),
])

raw_cust_path="/Volumes/ecommerce/source_data/raw/customers/*.csv"

dfcs=spark.read.option("header","true").option("delimiter",",").schema(customers_schema).csv(raw_cust_path)

dfcs=dfcs.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())

display(dfcs.limit(5))

dfcs.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
        .saveAsTable(f"{catalog_name}.bronze.brz_customers")

**Date**

In [0]:
#schema

date_schema=StructType([
    StructField("date",StringType(),True),
    StructField("year",IntegerType(),True),
    StructField("day_name",StringType(),True),
    StructField("quarter",IntegerType(),True),
    StructField("week_of_year",StringType(),True),
])

raw_date_path="/Volumes/ecommerce/source_data/raw/date/*.csv"

dfdate=spark.read.option("header","true").option("delimiter",",").schema(date_schema).csv(raw_date_path)

display(dfdate.limit(5))

dfdate=dfdate.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())

dfdate.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .saveAsTable(f"{catalog_name}.bronze.brz_date")